# Performance PR Analysis

Data: `full_analysis_distilled.csv` (repo root)  
Figures: `analysis_viz/figures/`


In [ ]:
# Setup
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline


def _find_csv() -> Path:
    """Locate full_analysis_distilled.csv from repo root or analysis_viz/."""
    search_roots = [
        Path.cwd(),
        Path.cwd() / "analysis_viz",
        Path.cwd().parent,
    ]
    for root in search_roots:
        candidate = (root / "full_analysis_distilled.csv").resolve()
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Missing full_analysis_distilled.csv. Run: python3 generate_full_analysis.py"
    )


def _notebook_dir() -> Path:
    cwd = Path.cwd()
    if cwd.name == "analysis_viz":
        return cwd
    if (cwd / "analysis_viz").is_dir():
        return cwd / "analysis_viz"
    return cwd


CSV_PATH = _find_csv()
FIG_DIR = _notebook_dir() / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "DejaVu Sans"],
    "axes.unicode_minus": False,
})

df = pd.read_csv(CSV_PATH)
df["is_merged"] = (df["status"] == "merged").astype(int)
df["outcome_label"] = df["status"].map({"merged": "Merged", "closed": "Closed", "open": "Open"})
terminal = df[df["status"].isin(["merged", "closed"])].copy()


def savefig(name: str):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print("saved", path)


def plot_merge_rate_by_bin(data, col, bins, labels, title, xlabel, fname):
    """Stacked bar (merged / not merged) + merge-rate line on twin axis."""
    COLOR_MERGED = "#4C78A8"
    COLOR_NOT_MERGED = "#F58518"
    COLOR_RATE = "#2F4B7C"
    COLOR_OVERALL = "#B279A2"

    sub = data[[col, "is_merged"]].dropna().copy()
    sub["bin"] = pd.cut(sub[col], bins=bins, labels=labels, include_lowest=True)
    grp = sub.groupby("bin", observed=True).agg(
        n=("is_merged", "size"),
        merged=("is_merged", "sum"),
    )
    grp["not_merged"] = grp["n"] - grp["merged"]
    grp["rate"] = grp["merged"] / grp["n"]

    fig, ax1 = plt.subplots(figsize=(10, 5))
    x = np.arange(len(grp))
    width = 0.62

    ax1.bar(
        x, grp["merged"], width,
        label="Merged", color=COLOR_MERGED, alpha=0.92, edgecolor="white", linewidth=0.8,
        zorder=2,
    )
    ax1.bar(
        x, grp["not_merged"], width, bottom=grp["merged"],
        label="Not merged", color=COLOR_NOT_MERGED, alpha=0.88, edgecolor="white", linewidth=0.8,
        zorder=2,
    )
    ax1.set_ylabel("PR count")
    ax1.set_xlabel(xlabel)
    ax1.set_title(title)
    ax1.set_xticks(x)
    ax1.set_xticklabels(grp.index.astype(str))
    ymax = max(grp["n"].max() * 1.18, 1)
    ax1.set_ylim(0, ymax)
    ax1.grid(axis="y", linestyle="--", alpha=0.35, zorder=0)
    ax1.set_axisbelow(True)

    for xi, (m, nm, total) in enumerate(zip(grp["merged"], grp["not_merged"], grp["n"])):
        ax1.text(xi, total + ymax * 0.02, f"n={int(total)}", ha="center", va="bottom", fontsize=8, color="#555")

    ax2 = ax1.twinx()
    ax2.plot(
        x, grp["rate"] * 100, color=COLOR_RATE, marker="o", markersize=7,
        linewidth=2.2, label="Merge rate (%)", zorder=4,
    )
    for xi, r in enumerate(grp["rate"] * 100):
        ax2.annotate(
            f"{r:.0f}%", (xi, r), textcoords="offset points", xytext=(0, 8),
            ha="center", fontsize=8, color=COLOR_RATE, fontweight="bold",
        )
    overall = terminal.is_merged.mean() * 100
    ax2.axhline(overall, color=COLOR_OVERALL, ls="--", lw=1.4, label=f"Overall ({overall:.0f}%)", zorder=1)
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("Merge rate (%)")

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=8, framealpha=0.95)
    savefig(fname)
    plt.show()

def plot_box_by_outcome(data, col, title, xlabel, fname, log_y=False):
    sub = data[[col, "outcome_label"]].dropna()
    order = ["Merged", "Closed", "Open"]
    groups = [sub.loc[sub.outcome_label == o, col].values for o in order if (sub.outcome_label == o).any()]
    labels = [o for o in order if (sub.outcome_label == o).any()]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    bp = ax.boxplot(groups, tick_labels=labels, patch_artist=True, showfliers=False)
    colors = {"Merged": "#4C78A8", "Closed": "#E45756", "Open": "#F58518"}
    for patch, lab in zip(bp["boxes"], labels):
        patch.set_facecolor(colors.get(lab, "#ccc"))
        patch.set_alpha(0.7)
    if log_y:
        ax.set_yscale("log")
    ax.set_title(title)
    ax.set_xlabel("Outcome")
    ax.set_ylabel(xlabel)
    savefig(fname)
    plt.show()


def explode_pipe_col(data, col, exclude=None):
    exclude = set(exclude or [])
    counter = {}
    for val in data[col].fillna(""):
        for item in str(val).split("|"):
            item = item.strip()
            if item and item not in exclude:
                counter[item] = counter.get(item, 0) + 1
    return pd.Series(counter).sort_values(ascending=False)


def plot_bar_series(series, title, xlabel, fname, top_n=15, color="#4C78A8"):
    s = series.head(top_n).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(s))))
    ax.barh(s.index.astype(str), s.values, color=color, alpha=0.88)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    savefig(fname)
    plt.show()


OUTCOME_ORDER = ["Merged", "Closed", "Open"]
OUTCOME_COLORS = {"Merged": "#4C78A8", "Closed": "#E45756", "Open": "#F58518"}

BOUNDARY_TAG_LABELS = {
    "technical_stack": "Technical stack\n(stack / framework depth)",
    "evidence_required": "Evidence required\n(benchmark / repro gap)",
    "process": "Process / workflow\n(review, scope, CI)",
    "unknown": "Unknown",
}


def plot_comment_distribution_by_outcome(data, fname):
    """Grouped bars by comment bin — clearer than boxplot when ~46% have zero comments."""
    sub = data[["comment_total", "outcome_label"]].dropna().copy()
    max_val = int(sub["comment_total"].max())
    labels = ["0", "1", "2-3", "4-9", "10+"]
    bins = [-0.1, 0.5, 1.5, 3.5, 9.5, max(max_val, 10) + 1]
    sub["bin"] = pd.cut(sub["comment_total"], bins=bins, labels=labels, include_lowest=True)
    counts = pd.crosstab(sub["bin"], sub["outcome_label"]).reindex(labels)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.25, 1]})
    x = np.arange(len(labels))
    width = 0.24
    for i, outcome in enumerate(OUTCOME_ORDER):
        if outcome not in counts.columns:
            continue
        vals = counts[outcome].fillna(0).values
        axes[0].bar(x + (i - 1) * width, vals, width, label=outcome, color=OUTCOME_COLORS[outcome], alpha=0.9)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].set_xlabel("Comments (review + PR)")
    axes[0].set_ylabel("PR count")
    axes[0].set_title("Comment volume by outcome")
    axes[0].legend(title="Outcome", fontsize=8)
    axes[0].grid(axis="y", linestyle="--", alpha=0.35)

    stats = []
    for outcome in OUTCOME_ORDER:
        s = sub.loc[sub.outcome_label == outcome, "comment_total"]
        if s.empty:
            continue
        stats.append(
            {
                "outcome": outcome,
                "median": s.median(),
                "p75": s.quantile(0.75),
                "zero_pct": (s == 0).mean() * 100,
            }
        )
    stat_df = pd.DataFrame(stats)
    y = np.arange(len(stat_df))
    axes[1].barh(
        y,
        stat_df["median"],
        color=[OUTCOME_COLORS[o] for o in stat_df["outcome"]],
        alpha=0.78,
        height=0.5,
    )
    for yi, row in enumerate(stat_df.itertuples()):
        axes[1].plot([row.median, row.p75], [yi, yi], color="#333", lw=2)
        axes[1].plot(row.p75, yi, "|", color="#333", ms=10)
        axes[1].text(
            row.p75 + 0.2,
            yi,
            f"median={row.median:.0f}  |  zero={row.zero_pct:.0f}%",
            va="center",
            fontsize=9,
        )
    axes[1].set_yticks(y)
    axes[1].set_yticklabels(stat_df["outcome"])
    axes[1].set_xlabel("Comments")
    axes[1].set_title("Median & zero-comment share")
    axes[1].set_xlim(0, max(sub["comment_total"].quantile(0.95), 4) * 1.4)

    zero_all = (sub["comment_total"] == 0).mean() * 100
    fig.suptitle(f"Collaboration intensity: {zero_all:.0f}% of PRs have no comments", fontsize=11, y=1.02)
    savefig(fname)
    plt.show()


def plot_antipattern_in_fix(data, fname):
    """Rare-event field: avoid 99% pie chart; show totals + enumerated cases."""
    fix_col = data["antipattern_in_fix"].fillna("none").astype(str).replace({"None": "none", "nan": "none"})
    n = len(data)
    none_n = int((fix_col == "none").sum())
    bad_n = n - none_n
    non_none = fix_col[fix_col != "none"].value_counts()

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.8), gridspec_kw={"width_ratios": [1, 1.35]})
    bars = ax0.bar(
        ["Clean fix", "New antipattern\nin fix"],
        [none_n, bad_n],
        color=["#72B7B2", "#E45756"],
        width=0.52,
    )
    ax0.set_ylabel("PR count")
    ax0.set_title(f"antipattern_in_fix (n={n})")
    for bar, v in zip(bars, [none_n, bad_n]):
        ax0.text(
            bar.get_x() + bar.get_width() / 2,
            v + n * 0.012,
            f"{v}\n({v / n * 100:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    ax0.set_ylim(0, n * 1.1)
    ax0.grid(axis="y", linestyle="--", alpha=0.35)

    if len(non_none):
        labels = non_none.index.astype(str).tolist()
        y = np.arange(len(labels))
        ax1.barh(y, non_none.values, color="#E45756", alpha=0.85, height=0.62)
        ax1.set_yticks(y)
        ax1.set_yticklabels(labels, fontsize=9)
        ax1.set_xlabel("Count")
        ax1.set_title(f"All {bad_n} rare cases (each n=1)")
        ax1.set_xlim(0, 1.65)
        if "pr_id" in data.columns:
            for yi, label in enumerate(labels):
                pids = data.loc[fix_col == label, "pr_id"].astype(str).tolist()
                ax1.text(1.05, yi, f"PR {', '.join(pids)}", va="center", fontsize=8, color="#555")
    else:
        ax1.axis("off")

    fig.suptitle("Did the follow-up fix introduce a new inefficiency / bug pattern?", fontsize=11, y=1.02)
    savefig(fname)
    plt.show()


def plot_boundary_tag_distribution(data, terminal_data, fname):
    """boundary_tag = which agent capability boundary this PR illustrates."""
    col = data["boundary_tag"].fillna("unknown")
    counts = col.value_counts()
    order = [o for o in ["technical_stack", "process", "evidence_required", "unknown"] if o in counts.index]
    colors = {
        "technical_stack": "#4C78A8",
        "process": "#F58518",
        "evidence_required": "#E45756",
        "unknown": "#BAB0AC",
    }
    merge_rates = data.groupby(col, observed=True)["is_merged"].mean().reindex(order)

    fig, ax1 = plt.subplots(figsize=(9.5, 4.8))
    x = np.arange(len(order))
    ax1.bar(
        x,
        [counts[o] for o in order],
        color=[colors.get(o, "#72B7B2") for o in order],
        alpha=0.88,
        width=0.55,
    )
    ax1.set_xticks(x)
    ax1.set_xticklabels([BOUNDARY_TAG_LABELS.get(o, o) for o in order], fontsize=9)
    ax1.set_ylabel("PR count")
    ax1.set_title("Agent capability boundary (boundary_tag)")
    ax1.grid(axis="y", linestyle="--", alpha=0.35)
    for xi, o in enumerate(order):
        ax1.text(xi, counts[o] + 10, f"n={counts[o]}", ha="center", fontsize=9)

    ax2 = ax1.twinx()
    ax2.plot(x, merge_rates.values * 100, color="#2F4B7C", marker="o", lw=2.2, markersize=7, label="Merge rate (%)")
    overall = terminal_data["is_merged"].mean() * 100
    ax2.axhline(overall, color="#B279A2", ls="--", lw=1.2, label=f"Overall ({overall:.0f}%)")
    for xi, r in enumerate(merge_rates.values * 100):
        ax2.annotate(f"{r:.0f}%", (xi, r), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, color="#2F4B7C")
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("Merge rate (%)")
    ax2.legend(loc="upper right", fontsize=8)

    savefig(fname)
    plt.show()


def require_ready():
    """Ensure the setup cell has been run before plotting."""
    if "df" not in globals():
        raise RuntimeError("Data not loaded. Run the setup cell or Run All first.")


print("Setup complete")
print(f"   CSV: {CSV_PATH}")
print(f"   PRs: {len(df)} | terminal merge rate: {terminal.is_merged.mean():.1%}")
print(f"   figures: {FIG_DIR.resolve()}")
df.head(3)

## 1. Code churn vs merge outcome

In [ ]:
require_ready()

plot_box_by_outcome(df, "changes", "Code churn by outcome", "changes", "01_changes_boxplot.png", log_y=True)
plot_merge_rate_by_bin(
    terminal, "changes",
    bins=[0, 100, 500, 2000, 10000, df["changes"].max() + 1],
    labels=["<=100", "101-500", "501-2k", "2k-10k", ">10k"],
    title="Merge rate by code-change size (terminal PRs)",
    xlabel="Changes bin",
    fname="02_changes_merge_rate.png",
)

## 2. File count vs merge outcome

In [ ]:
require_ready()

plot_box_by_outcome(df, "file_count", "File count by outcome", "file_count", "03_file_count_boxplot.png", log_y=True)
plot_merge_rate_by_bin(
    terminal, "file_count",
    bins=[0, 1, 5, 20, 100, terminal["file_count"].max() + 1],
    labels=["1", "2-5", "6-20", "21-100", ">100"],
    title="Merge rate by file count (terminal PRs)",
    xlabel="Files bin",
    fname="04_file_count_merge_rate.png",
)

## 3. PR lifespan vs merge outcome

In [ ]:
require_ready()

plot_box_by_outcome(df, "lifespan_hours", "Lifespan by outcome", "hours", "05_lifespan_boxplot.png", log_y=True)
plot_merge_rate_by_bin(
    terminal, "lifespan_hours",
    bins=[0, 1, 24, 168, terminal["lifespan_hours"].max() + 1],
    labels=["<1h", "1-24h", "1-7d", ">7d"],
    title="Merge rate by PR lifespan (terminal PRs)",
    xlabel="Lifespan bin",
    fname="06_lifespan_merge_rate.png",
)

## 4. Comment volume vs merge outcome

In [ ]:
require_ready()

plot_comment_distribution_by_outcome(df, "07_comment_total_boxplot.png")
plot_merge_rate_by_bin(
    terminal, "comment_total",
    bins=[-0.1, 0.5, 2.5, 9.5, terminal["comment_total"].max() + 1],
    labels=["0", "1-2", "3-9", ">=10"],
    title="Merge rate by comment volume (terminal PRs)",
    xlabel="Comments bin",
    fname="08_comment_merge_rate.png",
)

## 5. Antipatterns, detection, optimization, regression

In [ ]:
require_ready()

plot_bar_series(explode_pipe_col(df, "inefficiency_antipattern", exclude={"none", "unknown"}),
                "Inefficiency antipatterns", "PR count", "09_antipattern_bar.png", color="#E45756")
plot_bar_series(explode_pipe_col(df, "detection_method", exclude={"unknown", ""}),
                "Detection methods", "PR count", "10_detection_method_bar.png", color="#54A24B")
plot_bar_series(df["optimization_layer"].fillna("(missing)").value_counts(),
                "Optimization layer", "PR count", "11_optimization_layer_bar.png")
plot_bar_series(df["regression_handling"].fillna("(missing)").value_counts(),
                "Regression handling", "PR count", "12_regression_handling_bar.png", color="#B279A2")

## 6. Antipattern in fix

In [ ]:
require_ready()

plot_antipattern_in_fix(df, "13_antipattern_in_fix.png")

## 7. Agent, reproducibility, boundary tags

In [ ]:
require_ready()

fig, ax = plt.subplots(figsize=(5, 5))
vc = terminal["status"].value_counts()
ax.pie(vc.values, labels=[f"{k} ({v})" for k, v in vc.items()], autopct="%1.1f%%",
       colors=["#4C78A8", "#E45756"], startangle=90)
ax.set_title("Terminal outcomes")
savefig("14_outcome_pie.png")
plt.show()

agent_stats = df.groupby("agent").agg(n=("pr_id", "count"), merge_rate=("is_merged", "mean")).query("n >= 30").sort_values("merge_rate")
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(agent_stats.index, agent_stats["merge_rate"] * 100, color="#4C78A8")
ax.axvline(terminal.is_merged.mean() * 100, color="#E45756", ls="--", label="avg")
ax.set_xlabel("Merge rate (%)")
ax.set_title("Merge rate by agent (n>=30)")
ax.legend()
savefig("15_agent_merge_rate.png")
plt.show()

repro_cross = pd.crosstab(df["reproducibility"].fillna("unknown"), df["outcome_label"])
fig, ax = plt.subplots(figsize=(8, 4.5))
repro_cross.plot(kind="bar", stacked=True, ax=ax, color=["#4C78A8", "#E45756", "#F58518"])
ax.set_title("Reproducibility vs outcome")
plt.xticks(rotation=0)
savefig("16_reproducibility_outcome.png")
plt.show()

plot_boundary_tag_distribution(df, terminal, "17_boundary_tag_bar.png")
plot_merge_rate_by_bin(terminal, "review_count", bins=[-0.1, 0.5, 1.5, 3.5, terminal["review_count"].max() + 1],
                       labels=["0", "1", "2-3", ">=4"], title="Merge rate by review count (terminal PRs)",
                       xlabel="Reviews", fname="18_review_count_merge_rate.png")
print("Done. Figures:", FIG_DIR.resolve())